In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from pypolydim import polydim

from src.full_order.mesh import load_mesh
from src.full_order.assembly import (
    assemble_operators,
    assemble_control_matrix,
    assemble_dirichlet_and_source,
)
from src.full_order.solve import solve_otd

In [ ]:
# parametri: modifica questi valori per esplorare soluzioni diverse
mu1 = 12.0
mu2 = 2.5
mu_u = 0.99

with open('../configs/test1.yaml') as f:
    config = yaml.safe_load(f)

mesh_dir = '../' + config['mesh']['path']
boundary_markers = config['boundary_markers']
omega_obs_regions = config['problem']['omega_obs']
alpha = config['problem']['alpha']

In [ ]:
mesh_data = load_mesh(mesh_dir, boundary_markers)
print(f"Nh = {mesh_data['Nh']}")

operators = assemble_operators(mesh_data, omega_obs_regions)
dirichlet_data = assemble_dirichlet_and_source(mesh_data, omega_obs_regions)
C = assemble_control_matrix(mesh_data, operators['node_to_dof'], mu_u)

y_sol, p_sol, u_sol = solve_otd(operators, C, dirichlet_data, mu1, mu2, alpha)

print(f"y: min={y_sol.min():.4f}, max={y_sol.max():.4f}")
print(f"p: min={p_sol.min():.4f}, max={p_sol.max():.4f}")
print(f"u: min={u_sol.min():.4f}, max={u_sol.max():.4f}")

In [ ]:
# riporta la soluzione dai soli DOF liberi a tutti i nodi della mesh (per il plot)
mesh = mesh_data['mesh']
trial_dofs_data = mesh_data['trial_dofs_data']
u_D_y = dirichlet_data['u_D_y']
u_D_p = dirichlet_data['u_D_p']

assemble = polydim.pde_tools.assembler_utilities.pcc_2_d

sol_y = assemble.extract_solution_on_cell0_ds(mesh, trial_dofs_data, y_sol, u_D_y)
sol_p = assemble.extract_solution_on_cell0_ds(mesh, trial_dofs_data, p_sol, u_D_p)

y_plot = sol_y.numeric_solution
p_plot = sol_p.numeric_solution

In [ ]:
x_nodes = np.array([mesh.cell0_d_coordinate_x(i) for i in range(mesh.cell0_d_total_number())])
y_nodes = np.array([mesh.cell0_d_coordinate_y(i) for i in range(mesh.cell0_d_total_number())])

triangles = np.array([
    [mesh.cell2_d_vertex(t, 0), mesh.cell2_d_vertex(t, 1), mesh.cell2_d_vertex(t, 2)]
    for t in range(mesh.cell2_d_total_number())
])
triang = mtri.Triangulation(x_nodes, y_nodes, triangles)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
tc = ax.tricontourf(triang, y_plot, levels=200, cmap='jet')
plt.colorbar(tc, ax=ax, label='y')
ax.set_title(f'Stato y - mu1={mu1}, mu2={mu2}, mu_u={mu_u}')
ax.set_aspect('equal')

ax = axes[1]
tc = ax.tricontourf(triang, p_plot, levels=200, cmap='jet')
plt.colorbar(tc, ax=ax, label='p')
ax.set_title(f'Aggiunto p - mu1={mu1}, mu2={mu2}, mu_u={mu_u}')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()